In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import re

### Locate the original data file

I first check that the Companies House ZIP file is available in the raw data folder.

In [ ]:
original_data_folder = Path("../data/raw")
for file in original_data_folder.iterdir():
    print(file.name)

### Check the contents of the downloaded archive

Before reading the dataset, I check which file is stored inside the ZIP archive.

In [ ]:
companies_zip_file = original_data_folder / "BasicCompanyDataAsOneFile-2026-08-01.zip"

with zipfile.ZipFile(companies_zip_file, "r") as zip_file:
    files_inside_zip = zip_file.namelist()

files_inside_zip

### Preview the company data

I first load only the first 5 rows of the Companies House dataset.
The goal is to understand the structure and available columns before processing the full file.

In [ ]:
companies_preview = pd.read_csv(
    companies_zip_file,
    compression="zip",
    nrows=5,
    dtype={"CompanyNumber": str}
)

companies_preview.columns = companies_preview.columns.str.strip()

companies_preview
companies_preview

### Inspect the available columns

I check the column names to understand what company information is available and which fields may be useful later for sampling and company matching.

In [ ]:
companies_preview.columns.tolist()

### Clean the column names

Some column names contain extra spaces at the beginning or end. I remove these spaces so the fields can be referenced consistently during the analysis.

In [ ]:
companies_preview.columns = companies_preview.columns.str.strip()

companies_preview.columns.tolist()

### Select the most useful company information

The dataset contains many columns, but only some of them are relevant for this project. I keep the fields that can help identify a company, describe it, and later compare it with VAT information found from other sources..

In [ ]:
useful_columns = [
    "CompanyName",
    "CompanyNumber",
    "CompanyStatus",
    "CompanyCategory",
    "IncorporationDate",
    "RegAddress.AddressLine1",
    "RegAddress.AddressLine2",
    "RegAddress.PostTown",
    "RegAddress.County",
    "RegAddress.Country",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

companies_preview[useful_columns]

### Check the data types

I inspect how the fields were read by Python. Company identifiers should be treated as text rather than numbers, because they are identifiers and may contain letters or leading zeros.

In [ ]:
companies_preview[useful_columns].dtypes

### Treat the company number as an identifier

The company number was initially read as an integer. Since it is an identifier rather than a numeric value, I read it as text to preserve its original format.

In [ ]:
companies_preview = pd.read_csv(
    companies_zip_file,
    compression="zip",
    nrows=5,
    skipinitialspace=True,
    dtype={"CompanyNumber": str}
)

companies_preview

### Understand the company population

Before choosing a sample, I want to understand the companies available in the dataset. I count the total number of companies and check how they are distributed by company status.

Because the dataset is large, I read it in smaller parts instead of loading the entire file into memory.

In [ ]:
total_companies = 0
company_status_counts = {}

for company_data_part in pd.read_csv(
    companies_zip_file,
    compression="zip",
    usecols=["CompanyStatus"],
    skipinitialspace=True,
    chunksize=100_000
):
    total_companies += len(company_data_part)

    status_counts = company_data_part["CompanyStatus"].value_counts()

    for status, count in status_counts.items():
        company_status_counts[status] = (
            company_status_counts.get(status, 0) + count
        )

In [ ]:
print("Total companies:", total_companies)

company_status_summary = (
    pd.Series(company_status_counts)
    .sort_values(ascending=False)
)

company_status_summary


Most companies in the dataset are active. Since the use case concerns suppliers that are currently doing business, I use only companies with the status `Active` for the main sample.

I exclude companies in liquidation, administration, or proposed for strike-off because they are less representative of the target supplier population.

In [ ]:
active_companies = company_status_counts["Active"]

active_share = active_companies / total_companies * 100

print("Active companies:", active_companies)
print(f"Share of active companies: {active_share:.1f}%")

### Select a random sample of active companies

For the proof of concept, I select 100 companies at random from all active Companies House records.

The sample is selected before searching for VAT numbers, so companies are not chosen based on whether their VAT information is easy to find.

I use a fixed random seed so that the same sample can be reproduced.

This sample represents active Companies House companies, not necessarily the exact supplier mix of a mid-sized manufacturer. Without access to the customer's supplier population, I avoid making assumptions about its industry distribution.

In [ ]:
sample_size = 100
random_seed = 42

random_generator = np.random.default_rng(random_seed)

selected_company_positions = np.sort(
    random_generator.choice(
        active_companies,
        size=sample_size,
        replace=False
    )
)

selected_company_positions[:10]

### Extract the selected companies

I read the Companies House data in smaller parts and keep only active companies whose positions were selected for the sample.

I retain the fields that will be useful for company identification, VAT discovery, and later validation.

In [ ]:
columns_for_sample = [
    "CompanyName",
    "CompanyNumber",
    "CompanyStatus",
    "CompanyCategory",
    "IncorporationDate",
    "RegAddress.AddressLine1",
    "RegAddress.AddressLine2",
    "RegAddress.PostTown",
    "RegAddress.County",
    "RegAddress.Country",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

selected_companies = []
active_position = 0

for company_data_part in pd.read_csv(
    companies_zip_file,
    compression="zip",
    usecols=columns_for_sample,
    skipinitialspace=True,
    dtype={"CompanyNumber": str},
    chunksize=100_000
):

    active_company_data = company_data_part[
        company_data_part["CompanyStatus"] == "Active"
    ].copy()

    number_of_active_companies = len(active_company_data)

    positions_in_this_part = selected_company_positions[
        (selected_company_positions >= active_position)
        & (
            selected_company_positions
            < active_position + number_of_active_companies
        )
    ]

    if len(positions_in_this_part) > 0:

        positions_inside_part = (
            positions_in_this_part - active_position
        )

        selected_companies.append(
            active_company_data.iloc[positions_inside_part]
        )

    active_position += number_of_active_companies

In [ ]:
company_sample = pd.concat(
    selected_companies,
    ignore_index=True
)

print("Companies selected:", len(company_sample))

company_sample.head()

### Check the selected sample

I verify that the sample contains 100 unique companies and review a few records before using it for VAT discovery.

In [ ]:
print("Number of companies:", len(company_sample))
print(
    "Unique company numbers:",
    company_sample["CompanyNumber"].nunique()
)

company_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "IncorporationDate",
        "RegAddress.PostTown",
        "RegAddress.Country",
        "SICCode.SicText_1"
    ]
].head(10)

### Save the selected sample

I save the 100 selected companies before starting VAT discovery. This keeps the sample fixed and prevents later choices from being influenced by how easy or difficult a company's VAT number is to find.

In [ ]:
processed_data_folder = Path("../data/processed")

sample_file = processed_data_folder / "company_sample_100.csv"

company_sample.to_csv(
    sample_file,
    index=False
)

print("Sample saved:", sample_file)

### Understand the selected sample

Before starting VAT discovery, I review the sample to understand what kinds of companies it contains and whether important identification information is missing.

In [ ]:
company_sample["RegAddress.Country"].value_counts(dropna=False)

### Observation

The registered country field is not fully consistent across the sample. Some companies are recorded as `England`, others as `United Kingdom`, while 16 companies have no country value.

I keep these records in the sample because a missing country field does not by itself mean that the company is unsuitable for the analysis. Other address fields, such as postcode and town, may still help identify the company later.

### Review company types

I check the legal categories represented in the sample to see whether the random selection contains different types of active companies.

In [ ]:
company_sample["CompanyCategory"].value_counts()

### Review business activities

I inspect the SIC descriptions to understand the range of business activities represented in the random sample.

In [ ]:
company_sample["SICCode.SicText_1"].value_counts().head(15)

### Observation

The random sample contains companies from a wide range of business activities, including construction, real estate, manufacturing, engineering, IT and professional services.

I also found two useful data-quality nuances. An `Active` Companies House status does not necessarily mean that a company is actively trading, as the sample includes a company classified as `99999 - Dormant Company`. In addition, `None Supplied` appears as a SIC value rather than a missing value, so a simple null check can overstate data completeness.

I keep these records in the sample and treat them as part of the real-world uncertainty in the source data.

### Check missing identification information

Company name, company number and address information may later help confirm whether a discovered VAT number belongs to the correct company. I therefore check how complete these fields are in the sample.

In [ ]:
important_fields = [
    "CompanyName",
    "CompanyNumber",
    "RegAddress.AddressLine1",
    "RegAddress.PostTown",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

missing_information = (
    company_sample[important_fields]
    .isna()
    .sum()
)

missing_information

### Observation

The sample has complete company names and company numbers, while a small number of records have missing address information.

I keep these companies in the sample because incomplete address data is realistic and may affect how confidently a VAT number can be matched back to the correct company.

### Review company age


I look at the incorporation years to get a better sense of the sample and see whether it includes a mix of newer and older companies.

In [ ]:
company_sample["IncorporationDate"] = pd.to_datetime(
    company_sample["IncorporationDate"],
    format="%d/%m/%Y",
    errors="coerce"
)

In [ ]:
company_sample["IncorporationYear"] = (
    company_sample["IncorporationDate"].dt.year
)

company_sample["IncorporationYear"].describe()

### Observation

The sample includes both long-established and recently incorporated companies. Incorporation years range from 1962 to 2026, with a median of 2020.

This suggests that the random sample is not limited to either very new or very old businesses, although a large part of it consists of relatively recent companies.

## VAT discovery research

The next step is to investigate where UK VAT registration numbers can actually be discovered.

For each source, I want to understand:
- whether it contains VAT registration numbers;
- how reliably a VAT number can be linked to a specific company;
- what coverage it may provide;
- whether the source could realistically be used at scale;
- what can cause false positives or missing results.

### Initial source hypotheses

I identified several possible routes for VAT discovery:

1. **Official company websites**  
   VAT numbers may appear on legal pages, terms and conditions, invoices, ecommerce pages or other parts of a company's website.

2. **Government spending and procurement data**  
   Some public spending datasets contain both supplier names and VAT registration numbers.

3. **EORI numbers**  
   For UK VAT-registered businesses, the VAT number can be embedded in the EORI number, making EORI a possible source of VAT candidates.

4. **Bulk web data**  
   VAT numbers may exist across large numbers of webpages and documents, making web corpora potentially more suitable than crawling websites individually.

Each route will be tested separately rather than assumed to work.

### Prepare the VAT discovery results

I create a results table for the selected companies. For each company, I will record where I searched, any VAT candidate found, how it was verified, and the final decision.

In [ ]:
vat_results = company_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.AddressLine1",
        "RegAddress.PostTown",
        "RegAddress.PostCode"
    ]
].copy()

vat_results["Source"] = ""
vat_results["SourceURL"] = ""
vat_results["VATCandidate"] = ""
vat_results["HMRCValid"] = ""
vat_results["HMRCName"] = ""
vat_results["HMRCAddress"] = ""
vat_results["CompanyMatch"] = ""
vat_results["FinalStatus"] = "NOT CHECKED"
vat_results["Notes"] = ""

vat_results.head()

In [ ]:
results_folder = Path("../results")

vat_results_file = results_folder / "vat_discovery_results.csv"

vat_results.to_csv(
    vat_results_file,
    index=False
)

print("Results file created:", vat_results_file)

### TEST 1: Government spending data

I start by testing a public government spending dataset from DEFRA.

The dataset is useful for this experiment because it contains supplier names, supplier postcodes and VAT registration numbers. I want to measure how many usable VAT records it contains and whether any of the suppliers overlap with my random Companies House sample.

### Load the DEFRA dataset

The file could not be read using the default UTF-8 encoding. I therefore load it using Windows-1252 (`cp1252`), which correctly handles the characters in this dataset.

In [ ]:
defra_file = original_data_folder / "defra_spending_january_2026.csv"

defra_data = pd.read_csv(
    defra_file,
    encoding="cp1252",
    skipinitialspace=True
)

print("Rows:", len(defra_data))
print("Columns:", len(defra_data.columns))

defra_data.head()

### Inspect the DEFRA data

I first review the available columns and the VAT field before trying to match suppliers to the Companies House sample.

In [ ]:
defra_data.columns.tolist()

In [ ]:
defra_data.columns = defra_data.columns.str.strip()

defra_data.columns.tolist()

### Check VAT number availability

I check how often a VAT registration number is provided in the DEFRA dataset before using it as a discovery source.

In [ ]:
vat_column = "Vat Registration Num"

total_rows = len(defra_data)
rows_with_vat = defra_data[vat_column].notna().sum()

print("Total rows:", total_rows)
print("Rows with VAT:", rows_with_vat)
print(f"Share with VAT: {rows_with_vat / total_rows * 100:.1f}%")

In [ ]:
defra_data["Vat Registration Num"].value_counts(
    dropna=False
).head(20)

### Observation

Although 81% of the transaction rows contain a value in the VAT field, the numbers are not stored in a consistent format.

Some include the `GB` prefix, some contain spaces or punctuation, and some may not have the expected number of digits. Therefore, a populated VAT field does not necessarily mean that the value is immediately usable.

I normalize the values before measuring usable VAT coverage.

### Normalize the VAT numbers

I keep the original VAT value and create a separate normalized version.

For comparison purposes, I remove the `GB` prefix, spaces and punctuation, while keeping only the digits. I then check whether the resulting value contains 9 digits.

In [ ]:
def normalize_vat_number(vat_value):
    if pd.isna(vat_value):
        return None

    vat_as_text = str(vat_value).upper().strip()

    if vat_as_text.startswith("GB"):
        vat_as_text = vat_as_text[2:]

    digits_only = "".join(
        character for character in vat_as_text
        if character.isdigit()
    )

    return digits_only if digits_only else None


defra_data["NormalizedVAT"] = (
    defra_data["Vat Registration Num"]
    .apply(normalize_vat_number)
)

In [ ]:
defra_data[
    ["Vat Registration Num", "NormalizedVAT"]
].dropna().head(20)

### Check the normalized VAT format

After normalization, I check how many VAT values contain exactly 9 digits.

This is only a format check. A 9-digit value is still only a VAT candidate until it is verified against HMRC.

In [ ]:
defra_data["Has9DigitVAT"] = (
    defra_data["NormalizedVAT"]
    .str.fullmatch(r"\d{9}", na=False)
)

rows_with_9_digit_vat = defra_data["Has9DigitVAT"].sum()

print("Rows with a VAT value:", rows_with_vat)
print("Rows with a 9-digit VAT candidate:", rows_with_9_digit_vat)
print(
    f"Share of VAT values with 9 digits: "
    f"{rows_with_9_digit_vat / rows_with_vat * 100:.1f}%"
)

### Observation

Most populated VAT values become valid 9-digit candidates after normalization. Of the 746 rows containing VAT information, 729 (97.7%) have exactly 9 digits after removing prefixes, spaces and punctuation.

This suggests that inconsistent formatting is more common than structurally unusable VAT data in this source. However, a correct 9-digit format does not prove that the VAT number is valid or belongs to the expected company.

### Review VAT values that do not match the expected format

I inspect the VAT values that do not result in exactly 9 digits after normalization to understand what kinds of data-quality problems remain.

In [ ]:
unusual_vat_values = defra_data.loc[
    defra_data["Vat Registration Num"].notna()
    & ~defra_data["Has9DigitVAT"],
    [
        "Supplier",
        "Vat Registration Num",
        "NormalizedVAT"
    ]
]

unusual_vat_values

### Observation

The values that failed the 9-digit format check are not all the same type of error.

Some are foreign VAT numbers, such as values prefixed with `LU` or `IE`. Others appear incomplete or contain an unexpected number of digits. I also found a 12-digit GB value, which may represent a standard 9-digit UK VAT registration number followed by a branch or subsidiary identifier.

For the main UK analysis, I treat the standard 9-digit VAT registration number as the target format and keep non-standard values separate rather than automatically classifying them as invalid.

### Preserve the VAT country prefix

Because the dataset also contains foreign VAT numbers, I keep the country prefix separately instead of removing it completely during normalization. This helps distinguish UK VAT candidates from foreign VAT identifiers.

In [ ]:
def get_vat_prefix(vat_value):
    if pd.isna(vat_value):
        return None

    vat_as_text = str(vat_value).upper().strip()

    letters = "".join(
        character for character in vat_as_text
        if character.isalpha()
    )

    return letters[:2] if letters else None


defra_data["VATPrefix"] = (
    defra_data["Vat Registration Num"]
    .apply(get_vat_prefix)
)

In [ ]:
defra_data["VATPrefix"].value_counts(dropna=False)

In [ ]:
unique_suppliers = defra_data["Supplier"].nunique()

suppliers_with_vat_candidate = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        "Supplier"
    ]
    .nunique()
)

unique_vat_candidates = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        "NormalizedVAT"
    ]
    .nunique()
)

print("Unique suppliers:", unique_suppliers)
print(
    "Suppliers with a 9-digit VAT candidate:",
    suppliers_with_vat_candidate
)
print(
    "Unique 9-digit VAT candidates:",
    unique_vat_candidates
)

print(
    f"Supplier-level VAT candidate coverage: "
    f"{suppliers_with_vat_candidate / unique_suppliers * 100:.1f}%"
)

### Observation

The DEFRA dataset contains 468 unique suppliers. Of these, 333 have at least one 9-digit VAT candidate, giving a supplier-level VAT candidate coverage of 71.2%.

There are 308 unique VAT candidates for 333 suppliers with VAT information. This means that some VAT numbers are associated with more than one supplier name and should be investigated before treating the supplier-to-VAT relationship as one-to-one.

These figures measure candidate availability only. The VAT numbers have not yet been verified against HMRC.

### Check VAT numbers linked to multiple supplier names

Before using this source for company matching, I check whether the same VAT candidate is associated with multiple supplier names.

In [ ]:
supplier_vat_pairs = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        ["Supplier", "NormalizedVAT"]
    ]
    .drop_duplicates()
)

supplier_names_per_vat = (
    supplier_vat_pairs
    .groupby("NormalizedVAT")["Supplier"]
    .nunique()
    .sort_values(ascending=False)
)

supplier_names_per_vat.head(15)

In [ ]:
vat_numbers_with_multiple_names = supplier_names_per_vat[
    supplier_names_per_vat > 1
].index

multiple_name_cases = (
    supplier_vat_pairs[
        supplier_vat_pairs["NormalizedVAT"].isin(
            vat_numbers_with_multiple_names
        )
    ]
    .sort_values("NormalizedVAT")
)

multiple_name_cases.head(30)

### Observation

Several VAT candidates are associated with more than one supplier name.

Some cases appear to be simple naming variations, such as `RSK ADAS LTD`, `RSK ADAS LIMITED`, and `RSK ADAS Ltd`.

Other cases involve clearly different supplier names sharing the same VAT candidate. This is not necessarily a data error: UK companies can be registered as part of a VAT group, where multiple corporate entities use a single VAT registration number.

This means that VAT is a strong identifier for tax registration, but it does not always identify a single Companies House legal entity on its own. Additional company information may be needed when resolving VAT group members.

### Match the Companies House sample with DEFRA suppliers

I compare the 100 randomly selected companies with the suppliers found in the DEFRA dataset.

I start with simple exact matching after basic name cleaning. I prefer to miss a possible match rather than incorrectly assign a VAT number to the wrong company.

In [ ]:
def clean_company_name(company_name):
    if pd.isna(company_name):
        return None

    company_name = str(company_name).upper()

    company_name = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        company_name
    )

    company_name = re.sub(
        r"\s+",
        " ",
        company_name
    ).strip()

    return company_name

### Clean company names

I standardize company and supplier names before matching them. I remove punctuation and extra spaces and convert the text to uppercase.

In [ ]:

def clean_company_name(company_name):
    if pd.isna(company_name):
        return None

    company_name = str(company_name).upper()

    company_name = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        company_name
    )

    company_name = re.sub(
        r"\s+",
        " ",
        company_name
    ).strip()

    return company_name

### Apply the same cleaning rules

I apply the same name cleaning to both Companies House companies and DEFRA suppliers so they can be compared consistently.

In [ ]:
company_sample["CleanCompanyName"] = (
    company_sample["CompanyName"]
    .apply(clean_company_name)
)

defra_data["CleanSupplierName"] = (
    defra_data["Supplier"]
    .apply(clean_company_name)
)

### Keep suppliers with VAT candidates

I keep only DEFRA suppliers that have a 9-digit VAT candidate, since these are the records that could provide a useful company-to-VAT match.

In [ ]:
defra_suppliers_with_vat = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        [
            "Supplier",
            "CleanSupplierName",
            "Supplier Postcode",
            "NormalizedVAT"
        ]
    ]
    .drop_duplicates()
)

### Match the sample with DEFRA suppliers

I compare the cleaned company names with the cleaned DEFRA supplier names using exact matching.

I use a conservative approach because assigning a VAT number to the wrong company would be worse than missing a possible match.

In [ ]:
exact_matches = company_sample.merge(
    defra_suppliers_with_vat,
    left_on="CleanCompanyName",
    right_on="CleanSupplierName",
    how="inner"
)

print("Exact matches found:", len(exact_matches))

### Review the matches

I review any exact matches together with their postcodes and VAT candidates before deciding whether they are reliable.

In [ ]:
exact_matches[
    [
        "CompanyName",
        "CompanyNumber",
        "Supplier",
        "NormalizedVAT",
        "RegAddress.PostCode",
        "Supplier Postcode"
    ]
]

### Observation

No exact matches were found between the 100 Companies House companies and the DEFRA suppliers.

DEFRA contains useful VAT information for many of its own suppliers, but this individual dataset has very limited coverage for a random sample of UK companies. I therefore do not use it as the main VAT discovery source.

## TEST 2: Official company websites

Company websites are a possible VAT discovery source because businesses may publish their VAT registration number on legal pages, terms and conditions, contact pages or other parts of their website.

I test this source on companies from the random sample to understand how often an official website can be identified and whether it exposes a VAT number.

### Select companies for the website test

I use the first 20 companies from the previously generated random sample. I do not select companies based on whether they have a known website or VAT number.

In [ ]:
website_test_sample = company_sample.head(20).copy()

website_test_sample[
    ["CompanyName", "CompanyNumber", "RegAddress.PostCode"]
]

### Prepare the website search results

For each company, I record whether I found an official website, whether a VAT candidate was present, where it was found and whether it was later verified.

In [ ]:
website_results = website_test_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.PostCode"
    ]
].copy()

website_results["Website"] = ""
website_results["VATCandidate"] = ""
website_results["VATSourcePage"] = ""
website_results["HMRCVerified"] = ""
website_results["FinalStatus"] = "NOT CHECKED"

website_results

Company: AJ PARTITIONS AND CEILINGS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: ANDREWS RESIDENTIAL LIMITED
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: A website with a similar name was found, but its legal information refers to a different registered company, so I rejected it as the official website.


Company: AOB SUBSEA LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: ARLINGTON GROUP ASSET MANAGEMENT LIMITED
Official website: https://www.agam.co.uk/
VAT candidate: 69706407
Status: VAT CANDIDATE - FORMAT ISSUE
Notes: A VAT number was found in an official company document, but it contains only 8 digits and therefore requires further verification.

Company: ASTRAZENECA UK LIMITED
Official website: https://www.astrazeneca.co.uk/
VAT candidate: 582323642
Status: VAT CANDIDATE - NOT VERIFIED
Notes: A VAT number was found on an official AstraZeneca legal page and still needs to be verified against HMRC.

Company: AUDIO NUTRITION LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: AV GLOBAL HOLDINGS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: AVATAR INVESTMENTS LTD
Official website: Not confirmed
VAT candidate: Not found
Status: UNRESOLVED
Notes: The company name produced several ambiguous search results, so I could not confidently identify the correct official website.

Company: B & D ELIAS PROPERTIES LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: BLAIRGOWRIE EVANGELICAL CHURCH SCIO
Official website: https://www.hopeblair.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified, but no VAT registration number was found on the pages checked.

Company: BLOOMSBURY COURT INTERIORS LIMITED
Official website: https://www.bloomsburycourtinteriors.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was confirmed using the company name and company number, but no VAT registration number was found on the pages checked.

Company: BRACI1 LIMITED
Official website: https://www.braci.co/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The website identifies itself as Braci1 Limited and provides the same company number, but no VAT registration number was found.

Company: BURLINGTON WELLESLEY SEARCH LIMITED
Official website: https://www.burlingtonwellesleysearch.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: An official company website was identified, but no VAT registration number was found through the initial website search.

Company: CAD CONSULTANTS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: CANDEY LIMITED
Official website: https://www.candey.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was confirmed using the company name, company number and registered address, but no VAT registration number was found on the pages checked.

Company: CHARLIE PERKINS LIMITED
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search. The company is also recorded as dormant in its latest accounts.

Company: CHRONOBAND104 LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: CLARITY FINANCIAL LIMITED
Official website: https://clarityfin.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified, but the VAT number displayed on the site belongs to Sandringham Financial Partners Ltd, not Clarity Financial Limited, so I rejected it.

Company: CO BUILT FABRICATION LIMITED
Official website: https://www.co-built.net/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: A website corresponding to the Co-Built fabrication business was identified, but no VAT registration number was found through the initial website search.

Company: CUBAN BOXING ACADEMY CIC
Official website: https://www.cubanboxingacademy.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified using the company name and address, but no VAT registration number was found on the pages checked.

### Record the website search results

I organize the results of the manual website search in a structured table so I can compare the outcomes across the 20 companies.

In [ ]:
website_results_data = [
    {
        "Company": "AJ PARTITIONS AND CEILINGS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "ANDREWS RESIDENTIAL LIMITED",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "A similar website was found, but its legal information referred to a different registered company."
    },
    {
        "Company": "AOB SUBSEA LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "ARLINGTON GROUP ASSET MANAGEMENT LIMITED",
        "OfficialWebsite": "https://www.agam.co.uk/",
        "VATCandidate": "69706407",
        "Status": "VAT CANDIDATE - FORMAT ISSUE",
        "Notes": "VAT number found in an official company document, but it contains only 8 digits."
    },
    {
        "Company": "ASTRAZENECA UK LIMITED",
        "OfficialWebsite": "https://www.astrazeneca.co.uk/",
        "VATCandidate": "582323642",
        "Status": "VAT CANDIDATE - NOT VERIFIED",
        "Notes": "VAT number found on an official company legal page."
    },
    {
        "Company": "AUDIO NUTRITION LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "AV GLOBAL HOLDINGS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "AVATAR INVESTMENTS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "UNRESOLVED",
        "Notes": "The company name produced ambiguous search results, so the official website could not be confirmed."
    },
    {
        "Company": "B & D ELIAS PROPERTIES LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "BLAIRGOWRIE EVANGELICAL CHURCH SCIO",
        "OfficialWebsite": "https://www.hopeblair.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BLOOMSBURY COURT INTERIORS LIMITED",
        "OfficialWebsite": "https://www.bloomsburycourtinteriors.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BRACI1 LIMITED",
        "OfficialWebsite": "https://www.braci.co/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BURLINGTON WELLESLEY SEARCH LIMITED",
        "OfficialWebsite": "https://www.burlingtonwellesleysearch.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CAD CONSULTANTS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CANDEY LIMITED",
        "OfficialWebsite": "https://www.candey.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CHARLIE PERKINS LIMITED",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CHRONOBAND104 LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CLARITY FINANCIAL LIMITED",
        "OfficialWebsite": "https://clarityfin.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "A VAT number appeared on the website, but it belonged to another company, so it was rejected."
    },
    {
        "Company": "CO BUILT FABRICATION LIMITED",
        "OfficialWebsite": "https://www.co-built.net/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CUBAN BOXING ACADEMY CIC",
        "OfficialWebsite": "https://www.cubanboxingacademy.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    }
]

website_results = pd.DataFrame(website_results_data)

website_results

### Save the website search results

I save the manual website search results so they can be used later for the final analysis and VAT verification.

In [ ]:
website_results_file = results_folder / "website_search_results.csv"

website_results.to_csv(
    website_results_file,
    index=False
)

print("Website results saved:", website_results_file)

### Observation

Official websites could be identified for only part of the sample, which already limits the coverage of this discovery method.

Even when a website was found, VAT information was not always published. I also found a case where a VAT number shown on the correct website belonged to another company mentioned on the page, showing the risk of accepting VAT numbers without further verification.

## Verify VAT candidates with HMRC

Any VAT number found during discovery is treated only as a candidate until it is checked against HMRC.

For each candidate, I verify:
- whether the VAT number is currently valid;
- the registered business name;
- the registered address;
- whether these details correspond to the Companies House company.

Company: ARLINGTON GROUP ASSET MANAGEMENT LIMITED
Official website: https://www.agam.co.uk/
VAT candidate: 69706407
HMRC verification: Failed - incorrect format
Status: NOT VERIFIED - FORMAT ISSUE
Notes: The number is presented as a VAT registration number in an official company document, but HMRC requires a 9-digit UK VAT number and does not accept this 8-digit value.

Company: ASTRAZENECA UK LIMITED
Official website: https://www.astrazeneca.co.uk/
VAT candidate: 582323642
HMRC verification: Valid
HMRC business name: ASTRAZENECA UK LIMITED
HMRC address: 1 FRANCIS CRICK AVENUE, CB2 0AA, GB
Company match: Yes
Status: VERIFIED
Notes: The VAT number was found on the official company website and independently confirmed through HMRC. The registered business name returned by HMRC matches the target company.

Check the company address 

In [ ]:
company_sample[
    company_sample["CompanyName"] == "ASTRAZENECA UK LIMITED"
][
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.AddressLine1",
        "RegAddress.AddressLine2",
        "RegAddress.PostTown",
        "RegAddress.PostCode"
    ]
]

### Observation

The AstraZeneca VAT candidate was successfully verified. HMRC confirmed both the company name and registered address, matching the Companies House record.

## TEST 3: EORI as a VAT discovery route

I test whether publicly available EORI numbers can provide another route to VAT discovery.

For UK VAT-registered businesses, the first 9 digits of a GB EORI correspond to the VAT registration number. However, an EORI can also exist for a business that is not VAT registered.

I therefore treat any VAT number derived from an EORI as a candidate that still requires HMRC verification.

Company: AJ PARTITIONS AND CEILINGS LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: ANDREWS RESIDENTIAL LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: AOB SUBSEA LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: ARLINGTON GROUP ASSET MANAGEMENT LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: ASTRAZENECA UK LIMITED
EORI found: Not found through public web search
EORI valid: Not checked
Derived VAT candidate: Not counted
HMRC VAT verification: Already verified separately
Status: EORI NOT FOUND
Notes: The company's VAT number is already known and verified, but I did not derive an EORI from that VAT because this would not test EORI as a discovery source.

Company: AUDIO NUTRITION LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: AV GLOBAL HOLDINGS LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: AVATAR INVESTMENTS LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified with reasonable confidence. The company name is also ambiguous in web search.

Company: B & D ELIAS PROPERTIES LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: BLAIRGOWRIE EVANGELICAL CHURCH SCIO
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: BLOOMSBURY COURT INTERIORS LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: BRACI1 LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: BURLINGTON WELLESLEY SEARCH LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CAD CONSULTANTS LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CANDEY LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CHARLIE PERKINS LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CHRONOBAND104 LTD
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CLARITY FINANCIAL LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CO BUILT FABRICATION LIMITED
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

Company: CUBAN BOXING ACADEMY CIC
EORI found: Not found
EORI valid: Not checked
Derived VAT candidate: Not found
HMRC VAT verification: Not applicable
Status: EORI NOT FOUND
Notes: No public EORI number could be identified through the initial web search.

### Observation

EORI can help recover a VAT candidate when an EORI is found elsewhere on the open web, but it does not solve the original discovery problem by itself.

Any VAT candidate derived from an EORI would still need to be verified through HMRC before being accepted.

### Observation

I could not identify a public EORI number for any of the 20 companies tested.

This does not mean that these companies do not have an EORI. It means that EORI was not discoverable through the public web-search approach used in this test.

The official GB EORI service is primarily a verification tool: it requires an EORI number as input rather than allowing company-name search. Therefore, EORI can be useful as an adjacent identifier when it is already available from another source, but it did not work as a standalone discovery route in this sample.

## Proof of concept summary

I created a random sample of 100 active Companies House companies before attempting VAT discovery.

I tested public spending data against the full sample and used the first 20 companies for a more detailed website and EORI test.

The purpose was not to maximize the number of VAT numbers found, but to understand which discovery routes actually work and how reliable their results are.

### Results

- **Companies House sample:** 100 active companies
- **DEFRA:** 0 exact matches with the 100-company sample
- **Official websites tested:** 20 companies
- **Official websites identified:** 10
- **VAT candidates requiring investigation:** 2
- **Fully verified VAT numbers:** 1
- **EORI numbers discovered:** 0
- **Verified website discovery rate:** 1 / 20 = 5%

### False-positive rate

I only report a VAT number as found when it is confirmed through HMRC and the returned company identity matches the target Companies House record.

One VAT number met all of these conditions and no incorrect VAT number was accepted.

The observed false-positive rate among accepted results was therefore 0% (0 out of 1).

This result should be interpreted cautiously because the number of confirmed results is very small.

### What the proof of concept shows

The experiment shows that UK VAT numbers can be discovered from the open web, but no single source tested provides sufficient coverage on its own.

Official company websites can produce high-confidence results when VAT information is explicitly published, but website availability and VAT publication are limited.

Public spending data can contain useful supplier-to-VAT mappings, but coverage is specific to the organisations publishing the data.

EORI is potentially useful when already available, but I could not discover it reliably through public web search.

The main limitation is therefore discovery rather than verification: HMRC provides a strong way to verify a VAT candidate, but finding that candidate for an arbitrary company remains difficult.

## What I would do with real resources


With more time and resources, I would test the same approach on a much larger number of companies and combine several sources instead of relying on only one.


### 1. Test on a larger sample

My proof of concept was intentionally small. The first step would be to repeat the experiment on a larger sample of Companies House records to understand whether the results are consistent.

This would give a better estimate of real VAT discovery coverage.


### 2. Improve official website discovery

One of the main difficulties in my sample was finding the correct official website for each company.

I would use more information together, such as the company name, Companies House number, address and postcode, to make this match more reliable.

If the website could not be identified with enough confidence, I would leave the company unresolved rather than risk using the wrong website.


### 3. Search more sources

I would combine several discovery sources.

Official company websites would still be useful, especially legal pages, terms and conditions, contact pages and PDFs.

I would also test more government datasets and commercial company datasets. The DEFRA experiment showed that public datasets can contain useful VAT information, but their coverage can be limited to a specific group of suppliers.

With more infrastructure, I would also automate part of the website search and page checking instead of reviewing every company manually.


### 4. Verify every VAT candidate

Every VAT number found would still be treated as a candidate until it is checked through HMRC.

I would compare the business name and address returned by HMRC with the Companies House record.

If the VAT number is valid but the company information does not match clearly, I would reject the result or send it for manual review.

I would prefer a missing VAT number to an incorrect company-to-VAT match.


### 5. Cost and manual review

Based on this small experiment, I cannot calculate a reliable cost per company.

The cost would depend on how much of the process could be automated and how many companies would require manual review.

I would first run a larger pilot and measure the time and resources needed per company before estimating the cost of processing the full 40,000-company dataset.

### 6. What I think would break first

Based on my sample, I think the first major difficulty would be identifying the correct official website.

I found companies with no clear website and also a case where a website with a very similar company name belonged to a different legal entity.

Other problems would include websites that do not publish VAT information, outdated information, VAT groups and companies with no publicly discoverable VAT number.


### 7. What I would monitor

In production, I would mainly monitor:

* how many companies have a confidently identified official website;
* how many produce a VAT candidate;
* how many candidates pass HMRC verification;
* how many candidates are rejected because the company details do not match;
* how many cases require manual review.


### Conclusion

With more resources, I would expect coverage to improve, but I would not expect one source to solve the problem.

The most realistic approach would combine several discovery sources with strict HMRC verification and conservative company matching.

The main goal would be to improve coverage without increasing the risk of assigning the wrong VAT number to a company.


## Final conclusion

This proof of concept showed me that building a UK company-to-VAT dataset from open web sources is possible, but coverage is the main challenge.

The sources I tested had different limitations. DEFRA contained useful VAT information but had no overlap with my random Companies House sample. Official company websites produced one fully verified VAT number from the 20 companies tested, while other cases showed how easy it would be to create a wrong match. EORI was useful in theory, but I could not discover an EORI for any of the companies tested.

The most important lesson from the experiment is that finding a VAT candidate and proving that it belongs to the correct company are two different problems.

I would therefore use a conservative approach: combine several discovery sources, verify candidates through HMRC, and prefer a missing value when the company match is uncertain.